# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [59]:
import pandas as pd

PATH = '/content/product_catalog.csv'
cat = pd.read_csv(PATH)

original_row_count = len(cat)

print("Number of rows:", original_row_count)
display(cat.head())

Number of rows: 45


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [60]:
pop_mask = cat['product_name'].str.contains(
    'pop', case=False, na=False
)

pop_rows = cat.loc[pop_mask]

print("Count:", len(pop_rows))
display(pop_rows)

Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [61]:
strawberry_mask = pop_rows['product_name'].str.contains(
    'strawberry', case=False, na=False
)

strawberry_rows = pop_rows.loc[strawberry_mask]

print("Strawberry Pop-Tart variants:")
display(strawberry_rows[['sku', 'product_name']])

other_pop_tarts = pop_rows.loc[~strawberry_mask]

print("Pop-Tart rows that are a different product:")
display(other_pop_tarts[['sku', 'product_name']])

Strawberry Pop-Tart variants:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Pop-Tart rows that are a different product:


,sku,product_name


The strawberry Pop-Tart variants are PT-12 (pop-tarts strawberry), SKU#459812 (Strawberry Pop-Tart), POPTART-STRAW (Pop-Tarts Strawberry), and PT_STRAWBERRY (pop-tarts strawberry). These four SKUs appear to represent the same product and should be consolidated under one canonical SKU.

**Pop-Tart rows that are a different product:** None. All four matching rows are strawberry Pop-Tarts, so no other flavors need to be kept separate based on the current results.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [62]:
strawberry_skus = strawberry_rows['sku'].dropna().unique()

if len(strawberry_skus) == 0:
    raise ValueError("No strawberry Pop-Tart SKUs found. Check Q2.")

canonical_sku = strawberry_skus[0]

sku_map = {
    sku: canonical_sku
    for sku in strawberry_skus
    if sku != canonical_sku
}

cat['canonical_sku'] = cat['sku'].replace(sku_map)

print("Canonical SKU:", canonical_sku)
print("SKU mapping:", sku_map)

display(cat[['sku', 'canonical_sku', 'product_name']].head(20))

Canonical SKU: PT-12
SKU mapping: {'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}


,sku,canonical_sku,product_name
0,PT-12,PT-12,pop-tarts strawberry
1,SKU#459812,PT-12,Strawberry Pop-Tart
2,POPTART-STRAW,PT-12,Pop-Tarts Strawberry
3,PT_STRAWBERRY,PT-12,pop-tarts strawberry
4,WTR-24,WTR-24,BOTTLED WATER 24PK
5,WATER24PK,WATER24PK,Bottled Water 24-Pack
6,WTR_24PK,WTR_24PK,Water - 24 Pack
7,BATT-100,BATT-100,BATTERIES AA 4PK
8,BAT-AA4,BAT-AA4,AA Batteries 4-Pack
9,BATT_AA_4PK,BATT_AA_4PK,AA Batteries 4-Pack


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [63]:
print("Categories before normalization:")
print(cat['category'].value_counts(dropna=False))

cat['category'] = cat['category'].str.strip().str.title()

print("\nCategories after normalization:")
print(cat['category'].value_counts(dropna=False))

Categories before normalization:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

Categories after normalization:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks         

After normalization, capitalization differences are standardized, but some category labels still represent the same broader category. For example, Breakfast and Breakfast Foods, Grocery and Groceries, and Baby and Baby Products remain separate labels and may need to be consolidated.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [64]:
import re

def normalize_name(name):
    words = re.sub(r'[^\w\s]', '', str(name).lower()).split()
    return ' '.join(sorted(words))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

candidates = (
    cat.groupby('normalized_name')
    .agg(
        skus=('sku', list),
        product_names=('product_name', list),
        sku_count=('sku', 'nunique')
    )
    .reset_index()
)

candidates = candidates[candidates['sku_count'] > 1]

display(candidates)

,normalized_name,skus,product_names,sku_count
3,24pack beer light,"[BEER-24, LTBEER24]","[Light Beer (24-pack), Light Beer (24-pack)]",2
8,4pack aa batteries,"[BAT-AA4, BATT_AA_4PK]","[AA Batteries 4-Pack, AA Batteries 4-Pack]",2
12,board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Monopoly Board Game, Boa...",3
13,bread white,"[BREAD-WH, WHITEBREAD]","[WHITE BREAD, WHITE BREAD]",2
14,butter peanut,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]",2
15,candles emergency,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]",2
16,canned soup,"[SOUP-CAN, CNSOUP-01]","[Soup - Canned, Canned Soup]",2
17,cards playing,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]",2
20,chips pack variety,"[CHIPS-VP, CHIPVARIETY]","[Variety Pack - Chips, Chips Variety Pack]",2
22,duty heavy tarp,"[TARP-HD, HDTARP-01]","[Heavy Duty Tarp, TARP - HEAVY DUTY]",2


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [65]:
assert cat['canonical_sku'].notna().all(), \
    "Null canonical SKUs found!"

assert len(cat) == original_row_count, \
    "Row count changed!"

print("Passed: no null canonical SKUs and no rows added or removed.")

Passed: no null canonical SKUs and no rows added or removed.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Two fragmented product families are Monopoly board games and white bread. The Monopoly products appear under `BG-MONO`, `MONOPOLY-01`, and `GAME-MONO`, while white bread appears under `BREAD-WH` and `WHITEBREAD`. I would group SKUs that represent the same product despite differences in capitalization, punctuation, word order, or SKU naming. I would keep meaningful differences, such as size, flavor, or packaging, separate.

**b)** Consolidation becomes a business decision when product names alone do not establish whether two SKUs represent the same sellable product. For example, the Monopoly board games may have different editions or contents even though their names are similar. I would ask the product catalog manager or inventory team to confirm whether they should share one canonical SKU before consolidating them.